# v25 — Subject-Independent Stress Classification

**Diagnosis of v24 → LB 0.388:**
- Train (7 pids) and test (8 pids) share NO participants.
- v24's `StratifiedKFold` mixed pids across folds → model memorized per-nurse physiology → fake CV 0.82.
- `pid_enc` feature is pure noise on test.
- OOF threshold tuning was overfitting to leaky OOF.

**v25 fixes:**
- `LeaveOneGroupOut` CV by pid (7 folds) → honest subject-independent BA.
- Drop `pid_enc`.
- Per-pid z-score normalization of features (test nurses' raw values aren't out-of-distribution).
- Stronger regularization (smaller trees, higher reg_alpha/lambda).
- Tune alpha/t1/smooth on the HONEST grouped OOF.

Expected: honest CV will drop to ~0.42–0.50 but LB should track it. Goal: LB ≥ 0.40.


In [1]:
%pip -q install lightgbm scikit-learn pandas numpy scipy


Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats as spstats
from scipy import signal as sps
from scipy.integrate import trapezoid

from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold

import lightgbm as lgb

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR    = Path('.')
TRAIN_DATA  = pd.read_csv(DATA_DIR / 'train-sensor.csv')
TRAIN_LABEL = pd.read_csv(DATA_DIR / 'train-label.csv')
TEST_DATA   = pd.read_csv(DATA_DIR / 'test-sensor.csv')
TEST_LABEL  = pd.read_csv(DATA_DIR / 'test-label.csv')

print('Raw shapes')
print('  TRAIN_DATA :', TRAIN_DATA.shape)
print('  TRAIN_LABEL:', TRAIN_LABEL.shape)
print('  TEST_DATA  :', TEST_DATA.shape)
print('  TEST_LABEL :', TEST_LABEL.shape)
print('Train pids:', sorted(TRAIN_LABEL['pid'].astype(str).unique()))
print('Test  pids:', sorted(TEST_LABEL['pid'].astype(str).unique()))


Raw shapes
  TRAIN_DATA : (4694400, 8)
  TRAIN_LABEL: (815, 4)
  TEST_DATA  : (5921280, 8)
  TEST_LABEL : (1028, 4)
Train pids: ['43JW', 'C8Q6', 'DT5C', 'F1ZM', 'HDS9', 'P4DZ', 'TPQI']
Test  pids: ['01Z2', '2XO3', 'D1XP', 'NQRB', 'SE4Q', 'SNG7', 'TF0Y', 'Y21H']


In [3]:
SENSOR_COLS = ['accel_x', 'accel_y', 'accel_z', 'eda', 'heart_rate', 'temperature']

def clean_sensor(df):
    out = df.copy()
    out['pid'] = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    for c in SENSOR_COLS:
        out[c] = pd.to_numeric(out[c], errors='coerce').astype(float)
    out['accel_x'] = out['accel_x'].clip(-128, 127)
    out['accel_y'] = out['accel_y'].clip(-128, 127)
    out['accel_z'] = out['accel_z'].clip(-128, 127)
    out['eda']         = out['eda'].clip(0, 60)
    out['heart_rate']  = out['heart_rate'].clip(40, 190)
    out['temperature'] = out['temperature'].clip(20, 40)
    return out.sort_values(['pid', 'timestamp']).reset_index(drop=True)

def clean_label(df):
    out = df.copy()
    out['id']        = pd.to_numeric(out['id'], errors='raise').astype(int)
    out['pid']       = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    out['stress']    = pd.to_numeric(out['stress'], errors='coerce')
    return out

TRAIN_DATA  = clean_sensor(TRAIN_DATA)
TEST_DATA   = clean_sensor(TEST_DATA)
TRAIN_LABEL = clean_label(TRAIN_LABEL)
TEST_LABEL  = clean_label(TEST_LABEL)
print('Cleaned.')


Cleaned.


In [4]:
def compute_resting_baselines(sensor_df, low_pct=10):
    refs = {}
    for pid, grp in sensor_df.groupby('pid'):
        hr   = grp['heart_rate'].values.astype(float)
        eda  = grp['eda'].values.astype(float)
        temp = grp['temperature'].values.astype(float)
        valid = np.isfinite(hr) & np.isfinite(eda)
        if valid.sum() < 100:
            refs[pid] = {'hr': float(np.nanmedian(hr)) if valid.any() else 70.0,
                         'eda': float(np.nanmedian(eda)) if valid.any() else 1.0,
                         'temp': float(np.nanmedian(temp)) if valid.any() else 33.0,
                         'hr_std': 5.0, 'eda_std': 0.5,
                         'hr_median': float(np.nanmedian(hr)) if valid.any() else 70.0}
            continue
        hr_v, eda_v, temp_v = hr[valid], eda[valid], temp[valid]
        hr_z  = (hr_v  - hr_v.mean())  / (hr_v.std()  + 1e-9)
        eda_z = (eda_v - eda_v.mean()) / (eda_v.std() + 1e-9)
        arousal  = hr_z + eda_z
        thr      = np.percentile(arousal, low_pct)
        rest_mask = arousal < thr
        if rest_mask.sum() < 10:
            rest_mask = np.ones(len(arousal), dtype=bool)
        refs[pid] = {
            'hr':        float(np.median(hr_v[rest_mask])),
            'eda':       float(np.median(eda_v[rest_mask])),
            'temp':      float(np.median(temp_v[rest_mask])),
            'hr_std':    float(np.std(hr_v[rest_mask]) + 1e-3),
            'eda_std':   float(np.std(eda_v[rest_mask]) + 1e-3),
            'hr_median': float(np.median(hr_v)),
        }
    return refs

TRAIN_REFS = compute_resting_baselines(TRAIN_DATA)
TEST_REFS  = compute_resting_baselines(TEST_DATA)
print('Baselines computed. Train:', len(TRAIN_REFS), '| Test:', len(TEST_REFS))


Baselines computed. Train: 7 | Test: 8


In [5]:
WINDOW_MS = 180_000
HALF_MS   = 90_000
THIRD_MS  = 60_000
SHORT_MS  = 60_000
LONG_MS   = 300_000
XLONG_MS  = 600_000

def hrv_time_domain(bpm_series):
    f = {}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 10:
        for k in ['sdnn','rmssd','pnn25','pnn50','mean_rr','cv_rr']: f['hrv_'+k] = np.nan
        return f
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_diff = np.diff(rr)
    f['hrv_sdnn']    = float(np.std(rr))
    f['hrv_rmssd']   = float(np.sqrt(np.mean(rr_diff**2))) if len(rr_diff) else 0.0
    f['hrv_pnn25']   = float(np.mean(np.abs(rr_diff) > 25))*100 if len(rr_diff) else 0.0
    f['hrv_pnn50']   = float(np.mean(np.abs(rr_diff) > 50))*100 if len(rr_diff) else 0.0
    f['hrv_mean_rr'] = float(np.mean(rr))
    f['hrv_cv_rr']   = f['hrv_sdnn'] / f['hrv_mean_rr'] if f['hrv_mean_rr'] > 1e-6 else 0.0
    return f

def hrv_frequency_domain(bpm_series):
    f = {'hrv_vlf':np.nan,'hrv_lf':np.nan,'hrv_hf':np.nan,'hrv_lf_hf':np.nan,'hrv_total_power':np.nan}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 60: return f
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    if len(bpm_1hz) < 30: return f
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_c = rr - rr.mean()
    nperseg = min(len(rr_c), 64)
    if nperseg < 16: return f
    try:
        freqs, psd = sps.welch(rr_c, fs=1.0, nperseg=nperseg, noverlap=nperseg//2, scaling='density')
        def bp(lo, hi):
            mask = (freqs >= lo) & (freqs < hi)
            return float(trapezoid(psd[mask], freqs[mask])) if mask.sum() >= 2 else 0.0
        f['hrv_vlf'] = bp(0.0033, 0.04); f['hrv_lf'] = bp(0.04, 0.15); f['hrv_hf'] = bp(0.15, 0.40)
        f['hrv_total_power'] = f['hrv_vlf'] + f['hrv_lf'] + f['hrv_hf']
        f['hrv_lf_hf'] = f['hrv_lf'] / (f['hrv_hf'] + 1e-6)
    except: pass
    return f

def eda_peak_features(eda_series):
    f = {'eda_n_peaks':np.nan,'eda_peaks_per_min':np.nan,'eda_mean_prominence':np.nan,
         'eda_max_prominence':np.nan,'eda_mean_width':np.nan}
    eda = eda_series.dropna().values.astype(float)
    if len(eda) < 40: return f
    eda_4hz = eda[::8] if len(eda) >= 100 else eda
    if len(eda_4hz) < 16: return f
    try:
        wl = min(len(eda_4hz) - (1 if len(eda_4hz)%2==0 else 0), 15)
        if wl < 5: wl = 5
        if wl % 2 == 0: wl -= 1
        trend = sps.savgol_filter(eda_4hz, window_length=wl, polyorder=2) if len(eda_4hz)>20 else eda_4hz
        phasic = eda_4hz - trend + np.mean(eda_4hz)
        peaks, props = sps.find_peaks(phasic, prominence=0.02, distance=4, width=1)
        f['eda_n_peaks'] = float(len(peaks))
        dur_min = len(eda_4hz)/(4.0*60)
        f['eda_peaks_per_min'] = float(len(peaks)/dur_min) if dur_min > 0 else 0.0
        if len(peaks) > 0:
            f['eda_mean_prominence'] = float(np.mean(props['prominences']))
            f['eda_max_prominence']  = float(np.max(props['prominences']))
            f['eda_mean_width']      = float(np.mean(props['widths']))
        else:
            f['eda_mean_prominence'] = f['eda_max_prominence'] = f['eda_mean_width'] = 0.0
    except: pass
    return f

def eda_tonic_phasic_features(eda_series):
    f = {'eda_tonic_mean':np.nan,'eda_tonic_std':np.nan,'eda_phasic_mean':np.nan,
         'eda_phasic_std':np.nan,'eda_phasic_energy':np.nan,'eda_phasic_max':np.nan}
    eda = eda_series.dropna().values.astype(float)
    if len(eda) < 40: return f
    eda_4hz = eda[::8] if len(eda) >= 100 else eda
    if len(eda_4hz) < 20: return f
    try:
        wlen = min(len(eda_4hz) - (1 if len(eda_4hz)%2==0 else 0), 61)
        if wlen < 5: wlen = 5
        if wlen % 2 == 0: wlen -= 1
        tonic  = sps.savgol_filter(eda_4hz, window_length=wlen, polyorder=1)
        phasic = eda_4hz - tonic
        phasic_pos = np.maximum(phasic, 0)
        f['eda_tonic_mean']    = float(np.mean(tonic))
        f['eda_tonic_std']     = float(np.std(tonic))
        f['eda_phasic_mean']   = float(np.mean(phasic_pos))
        f['eda_phasic_std']    = float(np.std(phasic_pos))
        f['eda_phasic_energy'] = float(np.mean(phasic_pos**2))
        f['eda_phasic_max']    = float(np.max(phasic_pos))
    except: pass
    return f

def accel_jerk_features(ax, ay, az):
    f = {'accel_jerk_mean':np.nan,'accel_jerk_std':np.nan,'accel_jerk_max':np.nan}
    if len(ax) < 5: return f
    try:
        mag  = np.sqrt(ax.astype(float)**2 + ay.astype(float)**2 + az.astype(float)**2)
        mag  = mag[np.isfinite(mag)]
        if len(mag) < 5: return f
        jerk = np.abs(np.diff(mag))
        f['accel_jerk_mean'] = float(np.mean(jerk))
        f['accel_jerk_std']  = float(np.std(jerk))
        f['accel_jerk_max']  = float(np.max(jerk))
    except: pass
    return f

def timestamp_features(ts_ms):
    try:
        hour = (ts_ms / 3_600_000) % 24.0
        return {'hour_sin': float(np.sin(2*np.pi*hour/24)),
                'hour_cos': float(np.cos(2*np.pi*hour/24)),
                'hour_raw': float(hour)}
    except:
        return {'hour_sin':np.nan,'hour_cos':np.nan,'hour_raw':np.nan}


In [6]:
def extract_features(label_df, sensor_df, refs):
    """Same as v24 but drops pid_enc (was leakage on test)."""
    sensor_by_pid = {pid: grp.sort_values('timestamp').reset_index(drop=True)
                     for pid, grp in sensor_df.groupby('pid')}
    label_ts_by_pid = {pid: np.sort(grp['timestamp'].values.astype(float))
                       for pid, grp in label_df.groupby('pid')}
    rows = []
    for n, lrow in enumerate(label_df.itertuples(index=False), 1):
        pid = lrow.pid; ts = float(lrow.timestamp); lid = int(lrow.id)
        feat = {'id': lid, '_pid': pid}   # _pid kept for grouping; dropped before training
        sg = sensor_by_pid.get(pid)
        if sg is None:
            rows.append(feat); continue
        ta = sg['timestamp'].values
        wa     = sg.loc[(ta >= ts - WINDOW_MS) & (ta <= ts), SENSOR_COLS]
        wf     = sg.loc[(ta >= ts - WINDOW_MS) & (ta <  ts - HALF_MS),   SENSOR_COLS]
        wl     = sg.loc[(ta >= ts - HALF_MS)   & (ta <= ts),             SENSOR_COLS]
        wt1    = sg.loc[(ta >= ts - WINDOW_MS) & (ta <  ts - 2*THIRD_MS),SENSOR_COLS]
        wt3    = sg.loc[(ta >= ts - THIRD_MS)  & (ta <= ts),             SENSOR_COLS]
        wshort = sg.loc[(ta >= ts - SHORT_MS)  & (ta <= ts),             SENSOR_COLS]
        wlong  = sg.loc[(ta >= ts - LONG_MS)   & (ta <= ts),             SENSOR_COLS]
        wxlong = sg.loc[(ta >= ts - XLONG_MS)  & (ta <= ts),             SENSOR_COLS]

        feat['window_count'] = len(wa)
        feat['window_completeness'] = min(1.0, len(wa) / max(WINDOW_MS/1000, 1))

        for c in SENSOR_COLS:
            v   = wa[c].dropna().values.astype(float)
            vf  = wf[c].dropna().values.astype(float)
            vl  = wl[c].dropna().values.astype(float)
            vt1 = wt1[c].dropna().values.astype(float)
            vt3 = wt3[c].dropna().values.astype(float)
            if len(v) == 0:
                for s in ['mean','std','min','max','median','skew','kurt','range',
                          'q25','q75','iqr','delta','slope','t1_mean','t3_mean','t3t1']:
                    feat[f'{c}_{s}'] = np.nan
                continue
            feat[f'{c}_mean']    = float(np.mean(v))
            feat[f'{c}_std']     = float(np.std(v))
            feat[f'{c}_min']     = float(np.min(v))
            feat[f'{c}_max']     = float(np.max(v))
            feat[f'{c}_median']  = float(np.median(v))
            feat[f'{c}_skew']    = float(spstats.skew(v)) if len(v)>2 else 0.0
            feat[f'{c}_kurt']    = float(spstats.kurtosis(v)) if len(v)>2 else 0.0
            feat[f'{c}_range']   = float(np.max(v)-np.min(v))
            feat[f'{c}_q25']     = float(np.percentile(v,25))
            feat[f'{c}_q75']     = float(np.percentile(v,75))
            feat[f'{c}_iqr']     = feat[f'{c}_q75'] - feat[f'{c}_q25']
            feat[f'{c}_delta']   = float(np.mean(vl)-np.mean(vf)) if len(vf) and len(vl) else 0.0
            feat[f'{c}_slope']   = float(np.polyfit(np.linspace(0,1,len(v)),v,1)[0]) if len(v)>2 else 0.0
            feat[f'{c}_t1_mean'] = float(np.mean(vt1)) if len(vt1) else float(np.mean(v))
            feat[f'{c}_t3_mean'] = float(np.mean(vt3)) if len(vt3) else float(np.mean(v))
            feat[f'{c}_t3t1']    = feat[f'{c}_t3_mean'] - feat[f'{c}_t1_mean']

        ax, ay, az = wa['accel_x'].values, wa['accel_y'].values, wa['accel_z'].values
        if len(ax):
            mag = np.sqrt(ax**2+ay**2+az**2)
            feat['accel_mag_mean'] = float(np.mean(mag))
            feat['accel_mag_std']  = float(np.std(mag))
            feat['accel_mag_max']  = float(np.max(mag))
        else:
            feat['accel_mag_mean'] = feat['accel_mag_std'] = feat['accel_mag_max'] = np.nan
        feat.update(accel_jerk_features(ax, ay, az))
        feat.update(hrv_time_domain(wa['heart_rate']))
        feat.update(hrv_frequency_domain(wa['heart_rate']))
        feat.update(eda_peak_features(wa['eda']))
        feat.update(eda_tonic_phasic_features(wa['eda']))

        for c in ['heart_rate','eda']:
            vs = wshort[c].dropna().values.astype(float)
            if len(vs) == 0:
                feat[f'{c}_short_mean'] = feat[f'{c}_short_std'] = \
                feat[f'{c}_short_max']  = feat[f'{c}_short_slope'] = np.nan
                continue
            feat[f'{c}_short_mean']  = float(np.mean(vs))
            feat[f'{c}_short_std']   = float(np.std(vs))
            feat[f'{c}_short_max']   = float(np.max(vs))
            feat[f'{c}_short_slope'] = float(np.polyfit(np.linspace(0,1,len(vs)),vs,1)[0]) if len(vs)>2 else 0.0

        for c in ['heart_rate','eda','temperature']:
            vl2 = wlong[c].dropna().values.astype(float)
            if len(vl2) == 0:
                feat[f'{c}_long_mean'] = feat[f'{c}_long_std'] = feat[f'{c}_long_slope'] = np.nan
                continue
            feat[f'{c}_long_mean']  = float(np.mean(vl2))
            feat[f'{c}_long_std']   = float(np.std(vl2))
            feat[f'{c}_long_slope'] = float(np.polyfit(np.linspace(0,1,len(vl2)),vl2,1)[0]) if len(vl2)>2 else 0.0

        for c in ['heart_rate','eda']:
            v3 = wa[c].dropna().values.astype(float)
            v5 = wlong[c].dropna().values.astype(float)
            feat[f'{c}_3vs5min'] = float(np.mean(v3)-np.mean(v5)) if len(v3)>0 and len(v5)>0 else np.nan

        for c in ['temperature','heart_rate']:
            vxl = wxlong[c].dropna().values.astype(float)
            if len(vxl) > 2:
                feat[f'{c}_xlong_slope'] = float(np.polyfit(np.linspace(0,1,len(vxl)),vxl,1)[0])
                feat[f'{c}_xlong_mean']  = float(np.mean(vxl))
            else:
                feat[f'{c}_xlong_slope'] = feat[f'{c}_xlong_mean'] = np.nan

        ref = refs.get(pid, {})
        if ref:
            hr_m  = feat.get('heart_rate_mean', np.nan)
            eda_m = feat.get('eda_mean', np.nan)
            tmp_m = feat.get('temperature_mean', np.nan)
            feat['hr_dev_rest']      = (hr_m  - ref['hr'])  if np.isfinite(hr_m)  else np.nan
            feat['hr_dev_rest_std']  = (hr_m  - ref['hr'])  / ref['hr_std']  if np.isfinite(hr_m)  else np.nan
            feat['eda_dev_rest']     = (eda_m - ref['eda']) if np.isfinite(eda_m) else np.nan
            feat['eda_dev_rest_std'] = (eda_m - ref['eda']) / ref['eda_std'] if np.isfinite(eda_m) else np.nan
            feat['temp_dev_rest']    = (tmp_m - ref['temp']) if np.isfinite(tmp_m) else np.nan
            feat['compound_stress']  = feat['hr_dev_rest_std'] + feat['eda_dev_rest_std'] \
                                       if np.isfinite(feat['hr_dev_rest_std']) and np.isfinite(feat['eda_dev_rest_std']) else np.nan
            hr_med = ref.get('hr_median', ref['hr'])
            feat['hr_above_median'] = float(hr_m > hr_med) if np.isfinite(hr_m) else np.nan
            feat['hr_dev_median']   = (hr_m - hr_med)      if np.isfinite(hr_m) else np.nan
        else:
            for k in ['hr_dev_rest','hr_dev_rest_std','eda_dev_rest','eda_dev_rest_std',
                      'temp_dev_rest','compound_stress','hr_above_median','hr_dev_median']:
                feat[k] = np.nan

        try:
            hr_a  = wa['heart_rate'].dropna().values.astype(float)
            eda_a = wa['eda'].dropna().values.astype(float)
            tmp_a = wa['temperature'].dropna().values.astype(float)
            nm    = min(len(hr_a), len(eda_a), len(tmp_a))
            if nm >= 30:
                hr_a, eda_a, tmp_a = hr_a[:nm], eda_a[:nm], tmp_a[:nm]
                feat['corr_hr_eda']  = float(np.corrcoef(hr_a, eda_a)[0,1])  if hr_a.std()>1e-6 and eda_a.std()>1e-6  else 0.0
                feat['corr_hr_temp'] = float(np.corrcoef(hr_a, tmp_a)[0,1])  if hr_a.std()>1e-6 and tmp_a.std()>1e-6  else 0.0
                feat['corr_eda_temp']= float(np.corrcoef(eda_a,tmp_a)[0,1])  if eda_a.std()>1e-6 and tmp_a.std()>1e-6 else 0.0
            else:
                feat['corr_hr_eda'] = feat['corr_hr_temp'] = feat['corr_eda_temp'] = np.nan
        except:
            feat['corr_hr_eda'] = feat['corr_hr_temp'] = feat['corr_eda_temp'] = np.nan

        try:
            all_ts   = label_ts_by_pid.get(pid, np.array([ts]))
            prior_ts = all_ts[all_ts < ts]
            next_ts  = all_ts[all_ts > ts]
            feat['label_gap_prev_sec'] = float((ts - prior_ts[-1])/1000) if len(prior_ts)>0 else np.nan
            feat['label_gap_next_sec'] = float((next_ts[0]  - ts)/1000)  if len(next_ts)>0  else np.nan
        except:
            feat['label_gap_prev_sec'] = feat['label_gap_next_sec'] = np.nan

        feat.update(timestamp_features(ts))
        # NOTE: NO pid_enc — was leakage feature, killed for v25.
        rows.append(feat)
        if n % 200 == 0:
            print(f'  {n}/{len(label_df)}')
    return pd.DataFrame(rows).set_index('id')

print('Extracting train features...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA, TRAIN_REFS)
print('Extracting test features...')
test_features  = extract_features(TEST_LABEL,  TEST_DATA,  TEST_REFS)
print(f'train: {train_features.shape} | test: {test_features.shape}')


Extracting train features...
  200/815
  400/815
  600/815
  800/815
Extracting test features...
  200/1028
  400/1028
  600/1028
  800/1028
  1000/1028
train: (815, 166) | test: (1028, 166)


In [7]:
# ─── Per-pid z-score normalization ───
# For each numeric feature, subtract the pid's median and divide by the pid's IQR.
# This makes raw HR/EDA/temp scales comparable across nurses → test pids are
# no longer out-of-distribution on absolute values.
# Only normalize raw absolute features (mean/median/min/max/etc), NOT already-relative
# features (deltas, slopes, deviations from baseline, correlations, hour, hrv ratios).

DO_NOT_NORMALIZE_PREFIXES = (
    'hrv_', 'eda_phasic_', 'eda_tonic_',  # already pid-relative shape features
)
DO_NOT_NORMALIZE_EXACT = {
    'window_count', 'window_completeness',
    'hour_sin', 'hour_cos', 'hour_raw',
    'label_gap_prev_sec', 'label_gap_next_sec',
    'corr_hr_eda', 'corr_hr_temp', 'corr_eda_temp',
    'hr_dev_rest', 'hr_dev_rest_std',
    'eda_dev_rest', 'eda_dev_rest_std',
    'temp_dev_rest', 'compound_stress',
    'hr_above_median', 'hr_dev_median',
    'eda_n_peaks', 'eda_peaks_per_min',
    'eda_mean_prominence', 'eda_max_prominence', 'eda_mean_width',
}
SUFFIXES_RELATIVE = ('_delta', '_slope', '_t3t1', '_3vs5min', '_xlong_slope',
                     '_short_slope', '_long_slope', '_skew', '_kurt')

def is_normalizable(col):
    if col in DO_NOT_NORMALIZE_EXACT:
        return False
    if any(col.startswith(p) for p in DO_NOT_NORMALIZE_PREFIXES):
        return False
    if any(col.endswith(s) for s in SUFFIXES_RELATIVE):
        return False
    return True

def per_pid_zscore(features_df):
    """Add per-pid normalized versions of absolute features. Keeps originals."""
    out = features_df.copy()
    pid_col = features_df['_pid']
    norm_cols = [c for c in features_df.columns
                 if c != '_pid' and is_normalizable(c)
                 and pd.api.types.is_numeric_dtype(features_df[c])]
    new_cols = {}
    for c in norm_cols:
        # per-pid median and IQR
        grp_med = features_df.groupby('_pid')[c].transform('median')
        grp_q25 = features_df.groupby('_pid')[c].transform(lambda s: s.quantile(0.25))
        grp_q75 = features_df.groupby('_pid')[c].transform(lambda s: s.quantile(0.75))
        iqr = (grp_q75 - grp_q25).replace(0, np.nan)
        new_cols[f'{c}_pidz'] = (features_df[c] - grp_med) / iqr
    return pd.concat([out, pd.DataFrame(new_cols, index=out.index)], axis=1)

train_features_n = per_pid_zscore(train_features)
test_features_n  = per_pid_zscore(test_features)
print(f'After per-pid normalization: train {train_features_n.shape} | test {test_features_n.shape}')

# ─── Drop _pid (used only for grouping/normalization) and align columns ───
groups_train = train_features_n['_pid'].values  # save for GroupKFold
train_X_raw  = train_features_n.drop(columns=['_pid'])
test_X_raw   = test_features_n.drop(columns=['_pid'])

common_cols = [c for c in train_X_raw.columns if c in test_X_raw.columns]
train_X_raw = train_X_raw[common_cols]
test_X_raw  = test_X_raw[common_cols]
print(f'Common feature count: {len(common_cols)}')

# ─── Median imputation ───
imputer = SimpleImputer(strategy='median')
X_imp      = pd.DataFrame(imputer.fit_transform(train_X_raw),
                          columns=common_cols, index=train_X_raw.index)
X_test_imp = pd.DataFrame(imputer.transform(test_X_raw),
                          columns=common_cols, index=test_X_raw.index)

# ─── Targets and weights ───
tli = TRAIN_LABEL.set_index('id')
y   = tli.loc[X_imp.index, 'stress'].astype(int)

counts = Counter(y); total = len(y)
print('Class dist:', dict(counts))
class_weights = {0: total/(3*counts[0]), 1: total/(3*counts[1]), 2: total/(3*counts[2])}
sample_weights = np.array([class_weights[int(yi)] for yi in y])
train_prior    = np.array([counts[i]/total for i in range(3)])
print('Class weights:', {k: round(v,3) for k,v in class_weights.items()})
print('Train prior:', train_prior.round(3).tolist())
print('Unique groups (pids) in train:', len(np.unique(groups_train)))


After per-pid normalization: train (815, 252) | test (1028, 252)
Common feature count: 251
Class dist: {1: 66, 0: 162, 2: 587}
Class weights: {0: 1.677, 1: 4.116, 2: 0.463}
Train prior: [0.199, 0.081, 0.72]
Unique groups (pids) in train: 7


In [8]:
# More conservative than v24: smaller leaves, more reg, less aggressive tree growth.
# Rationale: with honest grouped CV, the model can't memorize per-nurse quirks,
# so we want it to find robust signal across nurses, not deep interactions.

LGBM_PARAMS = dict(
    n_estimators=2000,
    learning_rate=0.015,
    num_leaves=15,            # was 63 — much smaller, less memorization
    max_depth=5,              # explicit cap
    min_child_samples=25,     # was 20
    subsample=0.7,
    subsample_freq=1,
    colsample_bytree=0.5,
    reg_alpha=0.5,
    reg_lambda=1.0,
    class_weight='balanced',
    objective='multiclass',
    num_class=3,
    n_jobs=-1,
    verbose=-1,
)

SEEDS = [42, 7, 123, 17, 99]   # 5 seeds (was 10) — plenty given honest CV variance


In [9]:
# ──────────────────────────────────────────────────────────────────────
# LeaveOneGroupOut — each fold holds out ONE pid entirely.
# This is the honest subject-independent estimate of what LB will look like.
# ──────────────────────────────────────────────────────────────────────

logo = LeaveOneGroupOut()
oof_proba = np.zeros((len(X_imp), 3))
oof_count = np.zeros(len(X_imp))
fold_scores_per_seed = []

print('=== LeaveOneGroupOut (honest subject-independent OOF) ===')
print(f'Folds = unique pids = {len(np.unique(groups_train))}')

for seed in SEEDS:
    fold_scores = []
    for fold, (tr_idx, val_idx) in enumerate(logo.split(X_imp, y, groups=groups_train), 1):
        held_out_pid = np.unique(groups_train[val_idx])[0]
        model = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
        model.fit(
            X_imp.iloc[tr_idx], y.iloc[tr_idx],
            sample_weight=sample_weights[tr_idx],
            eval_set=[(X_imp.iloc[val_idx], y.iloc[val_idx])],
            callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(-1)],
        )
        val_proba = model.predict_proba(X_imp.iloc[val_idx])
        val_pred  = model.predict(X_imp.iloc[val_idx])
        score = balanced_accuracy_score(y.iloc[val_idx], val_pred)
        fold_scores.append((held_out_pid, score, dict(Counter(y.iloc[val_idx]))))
        oof_proba[val_idx] += val_proba
        oof_count[val_idx] += 1
    seed_mean = np.mean([s for _, s, _ in fold_scores])
    fold_scores_per_seed.append(seed_mean)
    print(f'  Seed {seed:4d}: LOPO mean BA = {seed_mean:.4f}')
    for pid_h, sc, dist in fold_scores:
        print(f'      held-out {pid_h:6s}: BA={sc:.4f}  val_dist={dist}')

oof_proba /= oof_count[:, None]
print(f'\n=== HONEST LOPO mean BA across seeds: {np.mean(fold_scores_per_seed):.4f} ===')
print(f'OOF argmax dist: {dict(Counter(oof_proba.argmax(1)))}')
honest_argmax_ba = balanced_accuracy_score(y.values, oof_proba.argmax(1))
print(f'OOF argmax BA  : {honest_argmax_ba:.4f}')


=== LeaveOneGroupOut (honest subject-independent OOF) ===
Folds = unique pids = 7
  Seed   42: LOPO mean BA = 0.4081
      held-out 43JW  : BA=0.1538  val_dist={2: 91, 0: 2}
      held-out C8Q6  : BA=0.3908  val_dist={2: 142, 0: 10}
      held-out DT5C  : BA=0.5431  val_dist={0: 58, 2: 18, 1: 14}
      held-out F1ZM  : BA=0.5510  val_dist={2: 134, 1: 3}
      held-out HDS9  : BA=0.3654  val_dist={0: 18, 2: 117}
      held-out P4DZ  : BA=0.3333  val_dist={1: 49, 0: 53, 2: 42}
      held-out TPQI  : BA=0.5194  val_dist={2: 43, 0: 21}
  Seed    7: LOPO mean BA = 0.4096
      held-out 43JW  : BA=0.1648  val_dist={2: 91, 0: 2}
      held-out C8Q6  : BA=0.3732  val_dist={2: 142, 0: 10}
      held-out DT5C  : BA=0.5555  val_dist={0: 58, 2: 18, 1: 14}
      held-out F1ZM  : BA=0.5510  val_dist={2: 134, 1: 3}
      held-out HDS9  : BA=0.3697  val_dist={0: 18, 2: 117}
      held-out P4DZ  : BA=0.3333  val_dist={1: 49, 0: 53, 2: 42}
      held-out TPQI  : BA=0.5194  val_dist={2: 43, 0: 21}
  Seed

In [10]:
# ──────────────────────────────────────────────────────────────────────
# Tune (alpha, t1, smooth) on the HONEST LOPO OOF. Now thresholds chosen
# here actually reflect what works on unseen pids.
# ──────────────────────────────────────────────────────────────────────

def threshold_predict(proba, t1=0.33, t0=None):
    pred = np.argmax(proba, axis=1).copy()
    if t1 < 0.50:
        override_1 = proba[:, 1] >= t1
        pred[override_1] = 1
    return pred

def make_session_groups(label_df, gap_ms=30*60*1000):
    labels = label_df.copy().reset_index(drop=True)
    labels['rowpos'] = np.arange(len(labels))
    out = []
    for pid, grp in labels.sort_values(['pid','timestamp']).groupby('pid', sort=False):
        ts   = grp['timestamp'].values.astype(float)
        sess = np.cumsum(np.r_[0, np.diff(ts) > gap_ms])
        for sid in np.unique(sess):
            out.append(grp['rowpos'].values[sess == sid])
    return out

def smooth_by_session(proba, sessions, strength=0.0):
    if strength <= 0: return proba.copy()
    out = proba.copy()
    for idx in sessions:
        mean = proba[idx].mean(axis=0, keepdims=True)
        out[idx] = (1 - strength) * proba[idx] + strength * mean
    return out

# Build session groups for OOF (matching X_imp's row order)
train_label_for_rows = TRAIN_LABEL.set_index('id').loc[X_imp.index].reset_index()
TRAIN_SESSIONS = make_session_groups(train_label_for_rows)

print('=== Honest grid search on LOPO OOF (alpha × t1 × smooth) ===')
best_ba   = -1
best_cfg  = None
y_arr = y.values
results = []

for alpha in [0.6, 0.8, 1.0, 1.2, 1.4]:
    cal = oof_proba * (train_prior ** alpha)
    cal = cal / cal.sum(axis=1, keepdims=True)
    for smooth in [0.0, 0.10, 0.20, 0.30]:
        sm = smooth_by_session(cal, TRAIN_SESSIONS, strength=smooth)
        for t1 in [0.33, 0.30, 0.27, 0.25, 0.22, 0.20, 0.17, 0.15, 0.12, 0.10, 0.08]:
            pred = threshold_predict(sm, t1)
            ba   = balanced_accuracy_score(y_arr, pred)
            results.append((alpha, t1, smooth, ba, dict(Counter(pred))))
            if ba > best_ba:
                best_ba, best_cfg = ba, (alpha, t1, smooth)

results.sort(key=lambda r: -r[3])
print(f'\nTop 15 (alpha, t1, smooth, BA, dist):')
for a, t, s, ba, d in results[:15]:
    print(f'  alpha={a:.1f} t1={t:.2f} smooth={s:.2f}  BA={ba:.4f}  dist={d}')

print(f'\n>>> BEST honest config: alpha={best_cfg[0]}, t1={best_cfg[1]}, smooth={best_cfg[2]}')
print(f'    Honest LOPO BA at best: {best_ba:.4f}')
print(f'    (vs argmax baseline: {honest_argmax_ba:.4f})')


=== Honest grid search on LOPO OOF (alpha × t1 × smooth) ===

Top 15 (alpha, t1, smooth, BA, dist):
  alpha=0.6 t1=0.17 smooth=0.30  BA=0.6267  dist={np.int64(1): 204, np.int64(0): 158, np.int64(2): 453}
  alpha=0.6 t1=0.20 smooth=0.30  BA=0.6234  dist={np.int64(1): 196, np.int64(0): 161, np.int64(2): 458}
  alpha=0.6 t1=0.20 smooth=0.20  BA=0.6209  dist={np.int64(1): 199, np.int64(0): 167, np.int64(2): 449}
  alpha=0.6 t1=0.15 smooth=0.20  BA=0.6195  dist={np.int64(1): 219, np.int64(0): 164, np.int64(2): 432}
  alpha=0.6 t1=0.15 smooth=0.30  BA=0.6194  dist={np.int64(1): 222, np.int64(0): 154, np.int64(2): 439}
  alpha=0.6 t1=0.12 smooth=0.20  BA=0.6185  dist={np.int64(1): 244, np.int64(0): 155, np.int64(2): 416}
  alpha=0.6 t1=0.17 smooth=0.20  BA=0.6179  dist={np.int64(1): 209, np.int64(0): 165, np.int64(2): 441}
  alpha=0.6 t1=0.12 smooth=0.30  BA=0.6150  dist={np.int64(1): 241, np.int64(0): 148, np.int64(2): 426}
  alpha=0.6 t1=0.20 smooth=0.10  BA=0.6110  dist={np.int64(1): 197, 

In [11]:
# ──────────────────────────────────────────────────────────────────────
# Final test predictions: train on ALL training pids per seed (no held-out
# pid at this stage). Use early stopping with a small in-fold validation set
# (one pid as ES validation) to pick #trees.
# ──────────────────────────────────────────────────────────────────────

print('=== Phase 2: Test inference ===')
all_test_proba = []
unique_pids = np.unique(groups_train)

for seed in SEEDS:
    rng = np.random.RandomState(seed)
    es_pid = rng.choice(unique_pids)            # one pid as ES validation
    es_mask = groups_train == es_pid
    tr_mask = ~es_mask

    model = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
    model.fit(
        X_imp[tr_mask], y[tr_mask],
        sample_weight=sample_weights[tr_mask],
        eval_set=[(X_imp[es_mask], y[es_mask])],
        callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(-1)],
    )
    proba = model.predict_proba(X_test_imp)
    all_test_proba.append(proba)
    print(f'  Seed {seed:4d} (ES on {es_pid}) → best_iter={model.best_iteration_}')

raw_test_proba = np.mean(all_test_proba, axis=0)
print('\nRaw test argmax dist:', dict(Counter(raw_test_proba.argmax(1))))


=== Phase 2: Test inference ===
  Seed   42 (ES on TPQI) → best_iter=438
  Seed    7 (ES on HDS9) → best_iter=740
  Seed  123 (ES on TPQI) → best_iter=375
  Seed   17 (ES on C8Q6) → best_iter=744
  Seed   99 (ES on C8Q6) → best_iter=817

Raw test argmax dist: {np.int64(2): 593, np.int64(0): 317, np.int64(1): 118}


In [12]:
# ──────────────────────────────────────────────────────────────────────
# Build several candidate submissions ranked by honest OOF BA.
# Submit them in the order printed below.
# ──────────────────────────────────────────────────────────────────────

TEST_SESSIONS = make_session_groups(TEST_LABEL)

def make_submission(proba, alpha, smooth_strength, t1, sessions, prior, fname):
    cal = proba * (prior ** alpha)
    cal = cal / cal.sum(axis=1, keepdims=True)
    cal = smooth_by_session(cal, sessions, strength=smooth_strength)
    preds = threshold_predict(cal, t1)
    pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': preds}).to_csv(fname, index=False)
    cnts  = np.bincount(preds, minlength=3)
    fracs = cnts / len(preds)
    return preds, cnts, fracs

# ─── Pick the top-N configs from honest grid search, plus safe fallbacks ───
top_configs = [(a, t, s) for a, t, s, ba, d in results[:5]]   # top-5 from grid

# Always include a few safe baselines as well
safe_configs = [
    (1.0, 0.33, 0.0),    # pure argmax, no calibration, no smoothing  (most conservative)
    (1.0, 0.33, 0.20),   # argmax + light session smoothing
    (0.8, 0.33, 0.20),   # mild prior dampening + smoothing  (matches your previous best style)
    (1.0, best_cfg[1], 0.0),  # honest-tuned t1 only
]

# Dedupe
seen = set()
final_configs = []
for cfg in top_configs + safe_configs:
    key = tuple(round(x, 3) for x in cfg)
    if key not in seen:
        seen.add(key)
        final_configs.append(cfg)

print(f'\n=== Generating {len(final_configs)} candidate submissions ===')
print(f'{"file":<55s} {"alpha":>5s} {"t1":>5s} {"smth":>5s} {"dist":>22s} {"oof_ba":>7s}')

# helper: estimate honest BA for each config from the OOF
def estimate_oof_ba(alpha, t1, smooth):
    cal = oof_proba * (train_prior ** alpha)
    cal = cal / cal.sum(axis=1, keepdims=True)
    sm = smooth_by_session(cal, TRAIN_SESSIONS, strength=smooth)
    return balanced_accuracy_score(y_arr, threshold_predict(sm, t1))

submission_records = []
for i, (a, t1, sm) in enumerate(final_configs, 1):
    fname = f'sub_v25_rank{i:02d}_a{a}_t{t1}_s{sm}.csv'
    preds, cnts, fracs = make_submission(raw_test_proba, a, sm, t1, TEST_SESSIONS, train_prior, fname)
    oof_ba = estimate_oof_ba(a, t1, sm)
    submission_records.append((fname, a, t1, sm, cnts.tolist(), oof_ba))
    print(f'{fname:<55s} {a:>5.2f} {t1:>5.2f} {sm:>5.2f} {str(cnts.tolist()):>22s} {oof_ba:>7.4f}')

# Sort by honest OOF BA descending and save canonical "best" copy
submission_records.sort(key=lambda r: -r[5])
import shutil
shutil.copy(submission_records[0][0], 'sub_v25_BEST_by_honest_oof.csv')
print(f'\n>>> sub_v25_BEST_by_honest_oof.csv  ←  copy of {submission_records[0][0]}')
print(f'    (honest OOF BA = {submission_records[0][5]:.4f})')



=== Generating 9 candidate submissions ===
file                                                    alpha    t1  smth                   dist  oof_ba
sub_v25_rank01_a0.6_t0.17_s0.3.csv                       0.60  0.17  0.30        [199, 139, 690]  0.6267
sub_v25_rank02_a0.6_t0.2_s0.3.csv                        0.60  0.20  0.30        [209, 118, 701]  0.6234
sub_v25_rank03_a0.6_t0.2_s0.2.csv                        0.60  0.20  0.20        [206, 121, 701]  0.6209
sub_v25_rank04_a0.6_t0.15_s0.2.csv                       0.60  0.15  0.20        [195, 149, 684]  0.6195
sub_v25_rank05_a0.6_t0.15_s0.3.csv                       0.60  0.15  0.30        [194, 148, 686]  0.6194
sub_v25_rank06_a1.0_t0.33_s0.0.csv                       1.00  0.33  0.00         [188, 51, 789]  0.3394
sub_v25_rank07_a1.0_t0.33_s0.2.csv                       1.00  0.33  0.20         [184, 47, 797]  0.3451
sub_v25_rank08_a0.8_t0.33_s0.2.csv                       0.80  0.33  0.20         [205, 60, 763]  0.3779
sub_v25_ran

In [13]:
print('\n========== v25 SUMMARY ==========')
print(f'Train pids: {sorted(np.unique(groups_train).tolist())}')
print(f'Test  pids: {sorted(TEST_LABEL["pid"].astype(str).unique().tolist())}')
print(f'Feature count : {X_imp.shape[1]}')
print(f'Train rows    : {len(y)}')
print(f'Honest LOPO BA (argmax)    : {honest_argmax_ba:.4f}')
print(f'Honest LOPO BA (best cfg)  : {best_ba:.4f}  ({best_cfg})')
print()
print('Key changes vs v24:')
print('  + LeaveOneGroupOut by pid (was StratifiedKFold → leakage)')
print('  + Per-pid z-score normalization of raw features')
print('  + Dropped pid_enc (was leakage)')
print('  + Stronger LGBM regularization (num_leaves 63→15, max_depth 5)')
print('  + No SMOTE (it was helping the leaky CV, not real generalization)')
print('  + Honest threshold tuning on LOPO OOF')
print()
print('SUBMISSION ORDER (by honest OOF BA, highest first):')
for i, (fname, a, t1, sm, cnts, ba) in enumerate(submission_records, 1):
    print(f'  {i}. {fname}  (honest BA={ba:.4f}, dist={cnts})')
print()
print('Submit  sub_v25_BEST_by_honest_oof.csv  FIRST.')
print('If it improves LB, try the next ranked one. If it does not, fall back to')
print('  sub_v25_rank01_a1.0_t0.33_s0.0.csv  (most conservative, pure argmax).')
print('==================================')



========== v25 SUMMARY ==========
Train pids: ['43JW', 'C8Q6', 'DT5C', 'F1ZM', 'HDS9', 'P4DZ', 'TPQI']
Test  pids: ['01Z2', '2XO3', 'D1XP', 'NQRB', 'SE4Q', 'SNG7', 'TF0Y', 'Y21H']
Feature count : 251
Train rows    : 815
Honest LOPO BA (argmax)    : 0.5916
Honest LOPO BA (best cfg)  : 0.6267  ((0.6, 0.17, 0.3))

Key changes vs v24:
  + LeaveOneGroupOut by pid (was StratifiedKFold → leakage)
  + Per-pid z-score normalization of raw features
  + Dropped pid_enc (was leakage)
  + Stronger LGBM regularization (num_leaves 63→15, max_depth 5)
  + No SMOTE (it was helping the leaky CV, not real generalization)
  + Honest threshold tuning on LOPO OOF

SUBMISSION ORDER (by honest OOF BA, highest first):
  1. sub_v25_rank01_a0.6_t0.17_s0.3.csv  (honest BA=0.6267, dist=[199, 139, 690])
  2. sub_v25_rank02_a0.6_t0.2_s0.3.csv  (honest BA=0.6234, dist=[209, 118, 701])
  3. sub_v25_rank03_a0.6_t0.2_s0.2.csv  (honest BA=0.6209, dist=[206, 121, 701])
  4. sub_v25_rank04_a0.6_t0.15_s0.2.csv  (honest BA=